In [ ]:
# =============================================================================
# TwoArm BOTH-EE v2 — search inside this cell for: POS_TOL_FINAL, PosTolRampTwoArm, w_mid, sum_progress
# (midpoint-biased targets + pos_tol ramp POS_TOL_START_TRAIN→POS_TOL_FINAL + dual-distance reward).
# =============================================================================
# %% TwoArm — dual SAC with **both** end-effectors within `POS_TOL` (same curriculum workspace as DualArm.ipynb)
# Run this notebook with the kernel working directory set to the `Two-Arm Robot` folder (where `TwoArms_col.urdf` lives).
import os
from pathlib import Path

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import pybullet as p
import pybullet_utils.bullet_client as bullet_client

import torch
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback, CallbackList
import matplotlib.pyplot as plt


def _find_urdf_dir() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "TwoArms_col.urdf").exists():
            return base
    return cwd


os.chdir(_find_urdf_dir())
print("Working directory:", Path.cwd())


def get_quaternion_from_euler(euler_angles):
    return p.getQuaternionFromEuler(euler_angles)


def get_angular_diff(q1, q2):
    q1, q2 = np.array(q1), np.array(q2)
    dot_prod = np.clip(np.abs(np.dot(q1, q2)), 0, 1)
    return 2 * np.arccos(dot_prod)


class DualArmRobotBothEEWithinTol(gym.Env):
    """Like `DualArmRobot` in DualArm.ipynb, but success requires **both** EEs within `pos_tol` of the target."""

    metadata = {"render_modes": ["human", "rgb_array"], "simulation_fps": 50}

    def __init__(
        self,
        render_mode="rgb_array",
        pos_tol=0.05,
        curriculum_total_timesteps=2_000_000,
        # Both-EE success needs more control steps than single-EE DualArm (320/250).
        episode_steps_early=450,
        episode_steps_late=400,
    ):
        super().__init__()
        self.render_mode = render_mode
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(20,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(45,), dtype=np.float32)

        self.pos_tol = float(pos_tol)
        self.orient_tol = 100.0
        self.episode_steps_early = int(episode_steps_early)
        self.episode_steps_late = int(episode_steps_late)
        self.curriculum_total_timesteps = int(curriculum_total_timesteps)
        self._global_env_steps = 0
        self._curriculum_d = 0.0

        conn_mode = p.DIRECT if render_mode != "human" else p.GUI
        self._bullet_client = bullet_client.BulletClient(connection_mode=conn_mode)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_number = 0
        self._bullet_client.resetSimulation()
        self._bullet_client.setGravity(0, 0, -9.8)

        self.robot_id = self._bullet_client.loadURDF("TwoArms_col.urdf", [0, 0, 0], useFixedBase=True)

        d = min(1.0, self._global_env_steps / float(max(1, self.curriculum_total_timesteps)))
        self._curriculum_d = float(d)
        self.max_step_size = int(
            round(
                float(
                    np.interp(
                        d,
                        [0.0, 1.0],
                        [float(self.episode_steps_early), float(self.episode_steps_late)],
                    )
                )
            )
        )
        self.max_step_size = max(1, self.max_step_size)

        x_lo = float(np.interp(d, [0.0, 1.0], [-0.02, -0.05]))
        x_hi = float(np.interp(d, [0.0, 1.0], [0.07, 0.10]))
        y_lo = float(np.interp(d, [0.0, 1.0], [0.33, 0.22]))
        y_hi = float(np.interp(d, [0.0, 1.0], [0.51, 0.62]))
        z_lo = float(np.interp(d, [0.0, 1.0], [0.17, 0.12]))
        z_hi = float(np.interp(d, [0.0, 1.0], [0.29, 0.34]))
        # Bias targets toward the midpoint between default-pose EEs so both arms can
        # plausibly reach the same point; blend toward uniform-in-box as curriculum advances.
        euler_target = np.random.uniform([-2.35, -0.78, -np.pi], [-0.78, 0.78, np.pi])
        self.quat_target = get_quaternion_from_euler(euler_target)

        sL0 = self._bullet_client.getLinkState(self.robot_id, 10)
        sR0 = self._bullet_client.getLinkState(self.robot_id, 20)
        mid = (np.array(sL0[0], dtype=np.float64) + np.array(sR0[0], dtype=np.float64)) * 0.5
        # Stronger midpoint bias longer → more reachable dual-EE goals.
        w_mid = float(np.interp(d, [0.0, 0.65, 1.0], [0.88, 0.42, 0.1]))
        # If the goal already contains both default EEs (large pos_tol + midpoint bias), SAC sees
        # length-1 "success" episodes (+220 only) — useless. Require a non-trivial start distance.
        margin = 0.02
        min_start_max_dist = float(self.pos_tol) + margin
        self.pos_target = mid.astype(np.float32)
        for _ in range(160):
            u = np.array(
                [
                    np.random.uniform(x_lo, x_hi),
                    np.random.uniform(y_lo, y_hi),
                    np.random.uniform(z_lo, z_hi),
                ],
                dtype=np.float64,
            )
            jitter = np.random.uniform(-0.08, 0.08, size=3)
            biased = mid + jitter
            cand = (1.0 - w_mid) * u + w_mid * biased
            cand[0] = np.clip(cand[0], x_lo, x_hi)
            cand[1] = np.clip(cand[1], y_lo, y_hi)
            cand[2] = np.clip(cand[2], z_lo, z_hi)
            self.pos_target = cand.astype(np.float32)
            o0 = self._get_obs()
            if max(float(o0[0]), float(o0[1])) >= min_start_max_dist:
                break
        else:
            corner = np.array([x_hi, y_lo, z_hi], dtype=np.float64)
            away = np.clip(0.35 * mid + 0.65 * corner, [x_lo, y_lo, z_lo], [x_hi, y_hi, z_hi])
            self.pos_target = away.astype(np.float32)

        obs = self._get_obs()
        dist_L, dist_R = float(obs[0]), float(obs[1])
        # Shape both arms: reward on the **bottleneck** (max distance), not only the nearest EE.
        self.prev_metric = max(dist_L, dist_R)
        self.prev_sum_dist = float(dist_L + dist_R)

        return obs, {}

    def _get_obs(self):
        state_L = self._bullet_client.getLinkState(self.robot_id, 10)
        state_R = self._bullet_client.getLinkState(self.robot_id, 20)

        pos_L, quat_L = np.array(state_L[0]), np.array(state_L[1])
        pos_R, quat_R = np.array(state_R[0]), np.array(state_R[1])

        dist_L = np.linalg.norm(pos_L - self.pos_target)
        dist_R = np.linalg.norm(pos_R - self.pos_target)

        q_diff_L = get_angular_diff(quat_L, self.quat_target)
        q_diff_R = get_angular_diff(quat_R, self.quat_target)

        joints = [self._bullet_client.getJointState(self.robot_id, i)[0] for i in range(20)]

        obs = np.concatenate(
            [
                [dist_L, dist_R],
                [q_diff_L, q_diff_R],
                self.pos_target,
                self.quat_target,
                pos_L,
                quat_L,
                pos_R,
                quat_R,
                joints,
            ]
        ).astype(np.float32)
        return obs

    def check_collisions(self):
        contact_points = self._bullet_client.getContactPoints(bodyA=self.robot_id)
        return len(contact_points)

    def set_curriculum_global_steps(self, n: int) -> None:
        self._global_env_steps = int(max(0, n))

    def step(self, action):
        self.step_number += 1

        scaled_action = action * np.pi
        scaled_action[0] = action[0] / 4.0
        scaled_action[10] = action[10] / 4.0
        scaled_action[3] = -scaled_action[2]
        scaled_action[13] = -scaled_action[12]

        self._bullet_client.setJointMotorControlArray(
            self.robot_id, list(range(20)), p.POSITION_CONTROL, targetPositions=scaled_action
        )
        self._bullet_client.stepSimulation()

        obs = self._get_obs()
        dist_L, dist_R = float(obs[0]), float(obs[1])
        q_diff_L, q_diff_R = obs[2], obs[3]

        current_max_dist = max(dist_L, dist_R)
        sum_dist = float(dist_L + dist_R)
        progress_reward = (self.prev_metric - current_max_dist) * 100.0
        sum_progress = (self.prev_sum_dist - sum_dist) * 15.0
        reward = progress_reward + sum_progress - 0.02 - 0.045 * sum_dist
        self.prev_metric = current_max_dist
        self.prev_sum_dist = sum_dist

        collisions = self.check_collisions()
        reward -= collisions * 0.5

        prox = max(dist_L, dist_R)
        soft = 1.35 * float(self.pos_tol)
        if prox < soft:
            reward += 0.25 * (1.0 - prox / max(soft, 1e-6))

        terminated = (dist_L < self.pos_tol) and (dist_R < self.pos_tol)
        if terminated:
            reward += 260.0

        truncated = self.step_number >= self.max_step_size

        info = {
            "is_success": terminated,
            "min_dist": float(min(dist_L, dist_R)),
            "max_dist": float(max(dist_L, dist_R)),
            "dist_error": float(max(dist_L, dist_R)),
            "dist_L": dist_L,
            "dist_R": dist_R,
            "q_diff_closest": float(min(float(q_diff_L), float(q_diff_R))),
            "collisions": collisions,
            "active_arm": "Left" if dist_L < dist_R else "Right",
            "curriculum_d": float(self._curriculum_d),
        }

        self._global_env_steps += 1
        return obs, reward, terminated, truncated, info


# pos_tol (m): both EEs within this distance of the same target. Rejection sampling avoids trivial 1-step success.
# HOLD: keep loose tolerance for early learning so successes can appear in SB3 success_rate; RAMP: then tighten.
POS_TOL_FINAL = 0.13
POS_TOL_START_TRAIN = 0.18
POS_TOL_HOLD_TIMESTEPS = 500_000
POS_TOL_RAMP_TIMESTEPS = 1_000_000


def _unwrap_twoarm_env(vec_env):
    inner = vec_env
    if hasattr(inner, "venv"):
        inner = inner.venv
    e0 = inner.envs[0]
    while hasattr(e0, "env"):
        e0 = e0.env
    return e0


class PosTolRampTwoArm(BaseCallback):
    # Hold start_tol, then linearly tighten to end_tol (avoids shrinking the goal while the policy is still weak).

    def __init__(
        self,
        start_tol: float,
        end_tol: float,
        hold_timesteps: int,
        ramp_timesteps: int,
        verbose: int = 0,
    ):
        super().__init__(verbose)
        self.start_tol = float(start_tol)
        self.end_tol = float(end_tol)
        self.hold_timesteps = int(max(0, hold_timesteps))
        self.ramp_timesteps = int(max(0, ramp_timesteps))
        self._n0 = None

    def _on_training_start(self) -> None:
        self._n0 = int(self.model.num_timesteps)

    def _on_step(self) -> bool:
        if self._n0 is None:
            self._n0 = int(self.model.num_timesteps)
        elapsed = max(0, int(self.model.num_timesteps) - self._n0)
        if elapsed < self.hold_timesteps:
            tol = self.start_tol
        elif self.ramp_timesteps <= 0:
            tol = self.end_tol
        else:
            t = elapsed - self.hold_timesteps
            alpha = min(1.0, t / float(self.ramp_timesteps))
            tol = self.start_tol + alpha * (self.end_tol - self.start_tol)
        _unwrap_twoarm_env(self.training_env).pos_tol = float(tol)
        return True


class PlottingCallbackTwoArmBothEE(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []
        self.final_dist_errors = []
        self.change_dist_errors = []
        self.final_q_errors = []
        self.change_q_errors = []
        self.success_rate = []
        self.recent_successes = []
        self.initial_dist = 0.0
        self.initial_qdiff = 0.0
        self.episode_timesteps = 0

    def _on_step(self) -> bool:
        info = self.locals["infos"][0]
        dist = float(info.get("dist_error", info.get("max_dist", 0.0)))
        if self.episode_timesteps == 0:
            self.initial_dist = dist
            self.initial_qdiff = float(info.get("q_diff_closest", 0.0))
        self.episode_timesteps += 1
        if self.locals.get("dones") is not None and self.locals["dones"][0]:
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])
            self.final_dist_errors.append(dist)
            self.change_dist_errors.append(dist - self.initial_dist)
            qdc = float(info.get("q_diff_closest", float("nan")))
            self.final_q_errors.append(qdc)
            self.change_q_errors.append(qdc - self.initial_qdiff)
            is_success = bool(info.get("is_success", False))
            self.recent_successes.append(1 if is_success else 0)
            if len(self.recent_successes) > 100:
                self.recent_successes.pop(0)
            self.success_rate.append(sum(self.recent_successes) / len(self.recent_successes))
            self.episode_timesteps = 0
        return True

    def _on_training_end(self) -> None:
        os.makedirs("matlab_data", exist_ok=True)
        np.savetxt("matlab_data/rewards.csv", np.array(self.episode_rewards), delimiter=",")
        np.savetxt("matlab_data/episode_lengths.csv", np.array(self.episode_lengths), delimiter=",")
        np.savetxt("matlab_data/final_distances.csv", np.array(self.final_dist_errors), delimiter=",")
        np.savetxt("matlab_data/change_distances.csv", np.array(self.change_dist_errors), delimiter=",")
        np.savetxt("matlab_data/final_quaternion_diff.csv", np.array(self.final_q_errors), delimiter=",")
        np.savetxt("matlab_data/change_quaternion_diff.csv", np.array(self.change_q_errors), delimiter=",")
        np.savetxt("matlab_data/success_rates.csv", np.array(self.success_rate), delimiter=",")
        last_ma = self.success_rate[-1] if self.success_rate else None
        print("Saved matlab_data/*.csv — latest 100-ep success MA (last row):", last_ma)

        n = min(
            len(self.episode_rewards),
            len(self.final_dist_errors),
            len(self.change_dist_errors),
            len(self.final_q_errors),
            len(self.change_q_errors),
            len(self.success_rate),
        )
        if n == 0:
            print("No completed episodes in this segment; skipping plots.")
            return
        ep = np.arange(1, n + 1)
        r = self.episode_rewards[:n]
        fd = self.final_dist_errors[:n]
        cd = self.change_dist_errors[:n]
        fq = self.final_q_errors[:n]
        cq = self.change_q_errors[:n]
        sr = self.success_rate[:n]
        blue = [0.15, 0.35, 0.75]
        lw = 0.9

        fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
        fig.suptitle(
            "TwoArm — both EEs within POS_TOL (curriculum workspace; bottleneck distance in plots)",
            fontsize=14,
        )

        axes[0, 0].plot(ep, r, color=blue, linewidth=lw)
        axes[0, 0].set_xlabel("Number of episodes")
        axes[0, 0].set_ylabel("Cumulative reward")
        axes[0, 0].set_title("Cumulative rewards vs episode number")
        axes[0, 0].grid(True, alpha=0.3)

        axes[0, 1].plot(ep, fd, color=blue, linewidth=lw)
        axes[0, 1].set_xlabel("Number of episodes")
        axes[0, 1].set_ylabel("Final max EE distance (m)")
        axes[0, 1].set_title("Final max(L,R) distance vs episode number")
        axes[0, 1].grid(True, alpha=0.3)

        axes[0, 2].plot(ep, cd, color=blue, linewidth=lw)
        axes[0, 2].set_xlabel("Number of episodes")
        axes[0, 2].set_ylabel("Change in max EE distance (m)")
        axes[0, 2].set_title("Change in max(L,R) distance vs episode number")
        axes[0, 2].grid(True, alpha=0.3)

        axes[1, 0].plot(ep, fq, color=blue, linewidth=lw)
        axes[1, 0].set_xlabel("Number of episodes")
        axes[1, 0].set_ylabel("Final quaternion difference (rad)")
        axes[1, 0].set_title("Final quaternion difference vs episode number")
        axes[1, 0].grid(True, alpha=0.3)

        axes[1, 1].plot(ep, cq, color=blue, linewidth=lw)
        axes[1, 1].set_xlabel("Number of episodes")
        axes[1, 1].set_ylabel("Change in quaternion difference (rad)")
        axes[1, 1].set_title("Change in quaternion difference vs episode number")
        axes[1, 1].grid(True, alpha=0.3)

        axes[1, 2].plot(ep, sr, color=[0.1, 0.55, 0.25], linewidth=1.1)
        axes[1, 2].set_xlabel("Number of episodes")
        axes[1, 2].set_ylabel("Success (0-1)")
        axes[1, 2].set_title("Success rate (100-episode moving average)")
        axes[1, 2].set_ylim(0, 1.05)
        axes[1, 2].grid(True, alpha=0.3)

        plt.show()


log_dir = "./logs/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs("./checkpoints/", exist_ok=True)


def make_env():
    env = DualArmRobotBothEEWithinTol(
        render_mode="direct",
        pos_tol=POS_TOL_START_TRAIN,
        curriculum_total_timesteps=2_000_000,
    )
    mon = os.devnull if os.environ.get("TWOARM_MONITOR_TO_NULL") == "1" else os.path.join(log_dir, "twoarm_both_ee_monitor")
    return Monitor(
        env,
        filename=mon,
        info_keywords=(
            "is_success",
            "dist_error",
            "max_dist",
            "dist_L",
            "dist_R",
            "q_diff_closest",
            "curriculum_d",
        ),
    )


env = DummyVecEnv([make_env])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)

policy_kwargs = dict(activation_fn=torch.nn.ReLU, net_arch=[256, 256, 256])

print("Initializing SAC (both-EE success)...")
model = SAC(
    "MlpPolicy",
    env,
    policy_kwargs=policy_kwargs,
    learning_rate=0.0003,
    batch_size=256,
    ent_coef="auto",
    verbose=1,
    device="auto",
)

TOTAL_TIMESTEPS = 2_000_000
print(f"Training {TOTAL_TIMESTEPS:,} steps | pos_tol hold {POS_TOL_START_TRAIN:g} for {POS_TOL_HOLD_TIMESTEPS:,}, ramp→{POS_TOL_FINAL:g} over {POS_TOL_RAMP_TIMESTEPS:,} | curriculum workspace...")
plot_cb = PlottingCallbackTwoArmBothEE()
pos_tol_cb = PosTolRampTwoArm(POS_TOL_START_TRAIN, POS_TOL_FINAL, POS_TOL_HOLD_TIMESTEPS, POS_TOL_RAMP_TIMESTEPS)
ckpt_cb = CheckpointCallback(
    save_freq=250_000,
    save_path="./checkpoints/",
    name_prefix="TwoArm_bothEE",
)
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=CallbackList([pos_tol_cb, ckpt_cb, plot_cb]),
    progress_bar=True,
)

_unwrap_twoarm_env(env).pos_tol = float(POS_TOL_FINAL)

model.save("TwoArm_BothEE_Curriculum")
env.save("vec_normalize_stats_twoarm_bothEE.pkl")
print("Done. Saved TwoArm_BothEE_Curriculum.zip (SB3) + vec_normalize_stats_twoarm_bothEE.pkl")
